---
title: "动态博弈树与 DFS：当经济学遇上计算机科学"
date: "2025-11-10"
excerpt: "博弈论提供了'灵魂'（决策逻辑），而数据结构提供了'肉体'（存储与运算实现）。本文用 Python 实现逆向归纳法，展示两者的完美交汇。"
tags: ["博弈论", "数据结构", "DFS", "逆向归纳法", "Python"]
category: ["经济学", "技术"]
language: "zh"
cover: "/images/covers/动态博弈树与 dfs.jpeg"
---

## 3. 实战：用 Python 构建博弈树

下面我们用 Python 手动构建一个简单的博弈树，并实现逆向归纳法求解。

### 3.1 定义博弈树的数据结构

In [7]:
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

@dataclass
class GameNode:
    """
    博弈树节点类
    
    属性:
        name: 节点名称（如 'A', 'B', 'C'）
        player: 当前决策的玩家（'Player1', 'Player2' 或 None 表示终点）
        children: 子节点列表
        actions: 指向各子节点的行动名称
        payoff: 终点的支付向量（如 (3, 2) 表示玩家 1 得 3，玩家 2 得 2）
        parent: 父节点（用于回溯）
    """
    name: str
    player: Optional[str] = None
    children: List['GameNode'] = field(default_factory=list)
    actions: List[str] = field(default_factory=list)  # 行动名称
    payoff: Optional[Tuple[int, int]] = None  # 终点支付
    parent: Optional['GameNode'] = None
    
    def is_terminal(self) -> bool:
        """判断是否为终点节点"""
        return len(self.children) == 0
    
    def add_child(self, child: 'GameNode', action: str):
        """添加子节点及对应的行动"""
        self.children.append(child)
        self.actions.append(action)
        child.parent = self

    def __repr__(self):
        if self.is_terminal():
            return f"Terminal({self.name}, payoff={self.payoff})"
        return f"Node({self.name}, player={self.player})"

### 3.2 构建一个简单的动态博弈：序贯讨价还价

考虑一个三阶段的序贯讨价还价博弈：
1. 玩家 1 先行动：选择"高要求"或"低要求"
2. 玩家 2 观察后行动：选择"接受"或"拒绝"
3. 如果拒绝，玩家 1 再次行动：选择"让步"或"坚持"

让我们用代码构建这个博弈树：

In [8]:
def build_bargaining_game() -> GameNode:
    """
    构建一个三阶段序贯讨价还价博弈树
    
    博弈结构:
    - 玩家 1：高要求 / 低要求
    - 玩家 2：接受 / 拒绝
    - 玩家 1：让步 / 坚持
    
    支付设定 (player1, player2):
    - 高要求 -> 接受：(8, 2)
    - 高要求 -> 拒绝 -> 让步：(4, 6)
    - 高要求 -> 拒绝 -> 坚持：(0, 0) 谈判破裂
    - 低要求 -> 接受：(4, 6)
    - 低要求 -> 拒绝 -> 让步：(5, 5)
    - 低要求 -> 拒绝 -> 坚持：(0, 0) 谈判破裂
    """
    # 根节点：玩家 1 的第一次决策
    root = GameNode(name="Root", player="Player1")
    
    # === 分支 1：玩家 1 选择"高要求" ===
    node_high = GameNode(name="High", player="Player2")
    root.add_child(node_high, action="高要求")
    
    # 玩家 2 选择"接受" -> 终点 (8, 2)
    terminal_accept_high = GameNode(name="Accept_High", payoff=(8, 2))
    node_high.add_child(terminal_accept_high, action="接受")
    
    # 玩家 2 选择"拒绝" -> 进入第三阶段
    node_reject_high = GameNode(name="Reject_High", player="Player1")
    node_high.add_child(node_reject_high, action="拒绝")
    
    # 玩家 1 选择"让步" -> (4, 6)
    terminal_concede_high = GameNode(name="Concede_High", payoff=(4, 6))
    node_reject_high.add_child(terminal_concede_high, action="让步")
    
    # 玩家 1 选择"坚持" -> (0, 0) 破裂
    terminal_insist_high = GameNode(name="Insist_High", payoff=(0, 0))
    node_reject_high.add_child(terminal_insist_high, action="坚持")
    
    # === 分支 2：玩家 1 选择"低要求" ===
    node_low = GameNode(name="Low", player="Player2")
    root.add_child(node_low, action="低要求")
    
    # 玩家 2 选择"接受" -> 终点 (4, 6)
    terminal_accept_low = GameNode(name="Accept_Low", payoff=(4, 6))
    node_low.add_child(terminal_accept_low, action="接受")
    
    # 玩家 2 选择"拒绝" -> 进入第三阶段
    node_reject_low = GameNode(name="Reject_Low", player="Player1")
    node_low.add_child(node_reject_low, action="拒绝")
    
    # 玩家 1 选择"让步" -> (5, 5)
    terminal_concede_low = GameNode(name="Concede_Low", payoff=(5, 5))
    node_reject_low.add_child(terminal_concede_low, action="让步")
    
    # 玩家 1 选择"坚持" -> (0, 0) 破裂
    terminal_insist_low = GameNode(name="Insist_Low", payoff=(0, 0))
    node_reject_low.add_child(terminal_insist_low, action="坚持")
    
    return root

# 构建博弈树
game_tree = build_bargaining_game()
print("博弈树构建完成！")
print(f"根节点：{game_tree}")
print(f"玩家 1 的行动：{game_tree.actions}")

博弈树构建完成！
根节点：Node(Root, player=Player1)
玩家 1 的行动：['高要求', '低要求']


### 3.3 可视化博弈树结构

In [9]:
def print_game_tree(node: GameNode, prefix: str = "", is_last: bool = True):
    """
    以树形结构打印博弈树
    """
    # 当前节点的标记
    connector = "└── " if is_last else "├── "
    
    if node.parent is None:
        # 根节点
        print(f"{node.name} [{node.player}]")
    else:
        # 找到对应的行动
        action_idx = node.parent.children.index(node)
        action = node.parent.actions[action_idx]
        
        if node.is_terminal():
            print(f"{prefix}{connector}【{action}】→ {node.name} (支付：{node.payoff})")
        else:
            print(f"{prefix}{connector}【{action}】→ {node.name} [{node.player}]")
    
    # 递归打印子节点
    if node.children:
        # 更新前缀
        if node.parent is None:
            new_prefix = prefix
        else:
            new_prefix = prefix + ("    " if is_last else "│   ")
        
        for i, child in enumerate(node.children):
            is_last_child = (i == len(node.children) - 1)
            print_game_tree(child, new_prefix, is_last_child)

print("\n=== 博弈树结构 ===")
print_game_tree(game_tree)


=== 博弈树结构 ===
Root [Player1]
├── 【高要求】→ High [Player2]
│   ├── 【接受】→ Accept_High (支付：(8, 2))
│   └── 【拒绝】→ Reject_High [Player1]
│       ├── 【让步】→ Concede_High (支付：(4, 6))
│       └── 【坚持】→ Insist_High (支付：(0, 0))
└── 【低要求】→ Low [Player2]
    ├── 【接受】→ Accept_Low (支付：(4, 6))
    └── 【拒绝】→ Reject_Low [Player1]
        ├── 【让步】→ Concede_Low (支付：(5, 5))
        └── 【坚持】→ Insist_Low (支付：(0, 0))


## 4. 逆向归纳法 vs. 深度优先搜索 (DFS)

这是两者最迷人的结合点：

* 博弈论里的**逆向归纳法 (Backward Induction)**，在计算机算法里本质上就是一种**带值回溯的深度优先搜索 (DFS)**。
* 为了求出子博弈精炼纳什均衡（SPE），程序会先递归到叶子节点（看结局），然后将结局的效用值一层层向上回传（Backpropagation），决定父节点的选择。

### 4.1 实现逆向归纳法

In [10]:
def backward_induction(node: GameNode) -> Tuple[Tuple[int, int], List[str]]:
    """
    逆向归纳法求解子博弈精炼纳什均衡 (SPE)
    
    参数:
        node: 当前博弈树节点
    
    返回:
        (payoff, equilibrium_path): 支付向量和均衡路径（行动列表）
    """
    # 基础情况：终点节点，直接返回支付
    if node.is_terminal():
        return (node.payoff, [])
    
    # 确定当前玩家的索引（用于从支付向量中选取自己的收益）
    player_idx = 0 if node.player == "Player1" else 1
    
    # 递归：对所有子节点进行逆向归纳
    best_payoff = None
    best_action = None
    best_subsequent_path = []
    
    for i, child in enumerate(node.children):
        # DFS：递归求解子博弈
        child_payoff, child_path = backward_induction(child)
        
        # 当前玩家选择最大化自己收益的行动
        current_player_payoff = child_payoff[player_idx]
        
        if best_payoff is None or current_player_payoff > best_payoff[player_idx]:
            best_payoff = child_payoff
            best_action = node.actions[i]
            best_subsequent_path = child_path
    
    # 构建完整的均衡路径
    equilibrium_path = [best_action] + best_subsequent_path
    
    return (best_payoff, equilibrium_path)

# 求解博弈
print("\n=== 逆向归纳法求解 ===")
spe_payoff, spe_path = backward_induction(game_tree)
print(f"子博弈精炼纳什均衡 (SPE) 结果:")
print(f"  均衡支付：Player1 = {spe_payoff[0]}, Player2 = {spe_payoff[1]}")
print(f"  均衡路径：{' → '.join(spe_path)}")


=== 逆向归纳法求解 ===
子博弈精炼纳什均衡 (SPE) 结果:
  均衡支付：Player1 = 4, Player2 = 6
  均衡路径：高要求 → 拒绝 → 让步


### 4.2 对比：标准的深度优先搜索 (DFS)

让我们看看标准的 DFS 遍历与逆向归纳法的区别：

In [11]:
def dfs_traversal(node: GameNode, prefix: str = "") -> List[str]:
    """
    标准的深度优先搜索遍历
    仅用于遍历所有节点，不进行决策优化
    """
    paths = []
    
    # 如果是叶子节点，返回当前路径
    if node.is_terminal():
        return [f"{prefix}{node.name} (支付：{node.payoff})"]
    
    # 递归访问所有子节点
    for i, child in enumerate(node.children):
        action = node.actions[i]
        child_prefix = f"{prefix}--[{action}]--> "
        child_paths = dfs_traversal(child, child_prefix)
        paths.extend(child_paths)
    
    return paths

print("\n=== DFS 遍历所有路径 ===")
all_paths = dfs_traversal(game_tree)
for i, path in enumerate(all_paths, 1):
    print(f"路径 {i}: {path}")


=== DFS 遍历所有路径 ===
路径 1: --[高要求]--> --[接受]--> Accept_High (支付：(8, 2))
路径 2: --[高要求]--> --[拒绝]--> --[让步]--> Concede_High (支付：(4, 6))
路径 3: --[高要求]--> --[拒绝]--> --[坚持]--> Insist_High (支付：(0, 0))
路径 4: --[低要求]--> --[接受]--> Accept_Low (支付：(4, 6))
路径 5: --[低要求]--> --[拒绝]--> --[让步]--> Concede_Low (支付：(5, 5))
路径 6: --[低要求]--> --[拒绝]--> --[坚持]--> Insist_Low (支付：(0, 0))


## 5. 进阶：带信息集的博弈

这是两者最大的不同点。在不完全信息博弈中，多个节点可能属于同一个"信息集"（玩家不知道自己处于哪个具体节点）。

* 在**数据结构**中，这相当于一种"模糊指针"或"并查集"的概念。
* 普通的计算机树通常是节点清晰的，而博弈树通过信息集引入了概率和不确定性。

### 5.1 扩展 GameNode 支持信息集

In [12]:
@dataclass
class GameNodeWithInfoSet(GameNode):
    """
    支持信息集的博弈树节点
    
    属性:
        info_set_id: 信息集 ID，同一信息集的节点无法被玩家区分
        belief: 玩家认为自己处于各节点的概率分布
    """
    info_set_id: Optional[str] = None
    belief: Dict[str, float] = field(default_factory=dict)

def build_poker_game_example() -> GameNodeWithInfoSet:
    """
    构建一个简单的扑克博弈示例（不完全信息）
    
    场景：
    1. 自然先发牌（高牌/低牌，玩家 1 知道，玩家 2 不知道）
    2. 玩家 1：下注 / 过牌
    3. 玩家 2（不知道牌面）：跟注 / 弃牌
    
    玩家 2 的两个决策节点属于同一个信息集
    """
    root = GameNodeWithInfoSet(name="Nature", player="Nature")
    
    # 自然行动：发高牌或低牌（等概率）
    node_high_card = GameNodeWithInfoSet(
        name="HighCard", player="Player1", 
        info_set_id=None,  # 玩家 1 知道自己拿到高牌
        belief={"HighCard": 1.0}
    )
    root.add_child(node_high_card, action="高牌 (50%)")
    
    node_low_card = GameNodeWithInfoSet(
        name="LowCard", player="Player1",
        info_set_id=None,  # 玩家 1 知道自己拿到低牌
        belief={"LowCard": 1.0}
    )
    root.add_child(node_low_card, action="低牌 (50%)")
    
    # 玩家 1 的行动
    for node in [node_high_card, node_low_card]:
        # 玩家 1 选择"下注"
        bet_node = GameNodeWithInfoSet(
            name=f"{node.name}_Bet", player="Player2",
            info_set_id="P2_after_bet",  # 玩家 2 的两个节点在同一信息集
            belief={"HighCard_Bet": 0.5, "LowCard_Bet": 0.5}  # 玩家 2 的信念
        )
        node.add_child(bet_node, action="下注")
        
        # 玩家 2 的行动
        terminal_fold = GameNodeWithInfoSet(name=f"{node.name}_Fold", payoff=(1, -1))
        bet_node.add_child(terminal_fold, action="弃牌")
        
        terminal_call = GameNodeWithInfoSet(name=f"{node.name}_Call", payoff=(2, -2) if node.name == "LowCard" else (-1, 1))
        bet_node.add_child(terminal_call, action="跟注")
        
        # 玩家 1 选择"过牌"
        check_terminal = GameNodeWithInfoSet(name=f"{node.name}_Check", payoff=(0, 0))
        node.add_child(check_terminal, action="过牌")
    
    return root

print("\n=== 带信息集的博弈树（扑克示例）===")
poker_game = build_poker_game_example()
print_game_tree(poker_game)


=== 带信息集的博弈树（扑克示例）===
Nature [Nature]
├── 【高牌 (50%)】→ HighCard [Player1]
│   ├── 【下注】→ HighCard_Bet [Player2]
│   │   ├── 【弃牌】→ HighCard_Fold (支付：(1, -1))
│   │   └── 【跟注】→ HighCard_Call (支付：(-1, 1))
│   └── 【过牌】→ HighCard_Check (支付：(0, 0))
└── 【低牌 (50%)】→ LowCard [Player1]
    ├── 【下注】→ LowCard_Bet [Player2]
    │   ├── 【弃牌】→ LowCard_Fold (支付：(1, -1))
    │   └── 【跟注】→ LowCard_Call (支付：(2, -2))
    └── 【过牌】→ LowCard_Check (支付：(0, 0))


## 5.2 信息集带来的计算挑战

信息集的引入虽然更符合现实，但也带来了巨大的计算挑战：

1. **信念更新**：玩家需要在博弈过程中不断更新对自然状态的信念（贝叶斯更新）
2. **策略空间爆炸**：每个信息集上的策略组合数量呈指数级增长
3. **均衡概念复杂化**：需要从纳什均衡升级到**贝叶斯纳什均衡**或**完美贝叶斯均衡**

## 5.3 从精确求解到近似求解

到这里，你可能已经意识到了一个问题：**逆向归纳法虽然精确，但只适用于小规模博弈树。**

让我们算一笔账：

| 博弈 | 状态空间大小 | 逆向归纳法可行性 |
|------|-------------|----------------|
| 三阶段讨价还价 | ~10 个节点 | 轻松求解 |
| 国际象棋 | $10^{43}$ 种局面 | 不可能 |
| 围棋 | $10^{170}$ 种局面 | 不可能 |
| 德州扑克 | $10^{160}$ 种信息集 | 不可能 |

这就是著名的**组合爆炸（Combinatorial Explosion）**问题。

### 那怎么办？

面对大规模博弈树，我们有两种选择：

1. **剪枝**：只搜索"看起来有希望"的分支（如 Alpha-Beta 剪枝）
2. **采样**：不遍历整棵树，而是随机采样部分路径进行估计

**蒙特卡洛树搜索（MCTS）** 就是第二种思路的代表。它放弃了"精确求解"，转而追求"在有限时间内找到足够好的解"。

> **关键洞察**：MCTS 的核心思想与博弈论的理性假设一脉相承——它通过大量随机模拟来估计每个行动的**期望效用**，然后选择期望效用最高的行动。这本质上是在用**统计方法逼近逆向归纳法**的结果。

## 6. 应用：蒙特卡洛树搜索 (MCTS) 简化版

AlphaGo 的核心算法 MCTS 是在博弈树上进行随机模拟，然后用模拟结果更新节点价值。下面是简化版的实现：

In [6]:
import random
from typing import Callable

class MCTSNode:
    """MCTS 节点"""
    def __init__(self, state, player: str, parent=None, action=None):
        self.state = state  # 游戏状态
        self.player = player  # 当前玩家
        self.parent = parent
        self.action = action  # 从父节点到当前节点的行动
        self.children = []
        self.visits = 0
        self.wins = 0
        self.untried_actions = []  # 尚未探索的行动
    
    def uct_score(self, c: float = 1.41) -> float:
        """
        UCT (Upper Confidence Bound for Trees) 分数
        用于平衡探索 (exploration) 和利用 (exploitation)
        """
        if self.visits == 0:
            return float('inf')  # 未访问节点优先探索
        exploitation = self.wins / self.visits
        exploration = c * (2 * (self.parent.visits if self.parent else 1) / self.visits) ** 0.5
        return exploitation + exploration
    
    def best_child(self) -> 'MCTSNode':
        """选择 UCT 分数最高的子节点"""
        return max(self.children, key=lambda x: x.uct_score())


def mcts(root_state: any, player: str, num_iterations: int = 1000,
       get_actions: Callable = None, apply_action: Callable = None,
       is_terminal: Callable = None, get_payoff: Callable = None) -> str:
    """
    简化版蒙特卡洛树搜索
    
    参数:
        root_state: 初始游戏状态
        player: 当前决策玩家
        num_iterations: 模拟次数
        get_actions: 获取合法行动的函数
        apply_action: 应用行动并返回新状态的函数
        is_terminal: 判断是否终点的函数
        get_payoff: 获取终点评定的函数
    
    返回:
        最佳行动
    """
    root = MCTSNode(root_state, player)
    root.untried_actions = get_actions(root_state)
    
    for _ in range(num_iterations):
        node = root
        state = root_state
        current_player = player
        
        # 1. 选择 (Selection): 用 UCT 选择子节点直到叶子节点
        while node.untried_actions == [] and node.children != []:
            node = node.best_child()
            state = apply_action(state, node.action)
            current_player = "Player2" if current_player == "Player1" else "Player1"
        
        # 2. 扩展 (Expansion): 如果有未尝试的行动，扩展一个新节点
        if node.untried_actions:
            action = random.choice(node.untried_actions)
            node.untried_actions.remove(action)
            new_state = apply_action(state, action)
            new_player = "Player2" if current_player == "Player1" else "Player1"
            new_node = MCTSNode(new_state, new_player, parent=node, action=action)
            new_node.untried_actions = get_actions(new_state)
            node.children.append(new_node)
            node = new_node
            state = new_state
        
        # 3. 模拟 (Rollout): 随机模拟到游戏结束
        rollout_player = current_player
        while not is_terminal(state):
            actions = get_actions(state)
            action = random.choice(actions)
            state = apply_action(state, action)
            rollout_player = "Player2" if rollout_player == "Player1" else "Player1"
        
        # 4. 回传 (Backpropagation): 将结果回传给路径上的所有节点
        payoff = get_payoff(state)
        while node is not None:
            node.visits += 1
            # 如果该节点玩家与根节点玩家相同，则累加其收益
            if node.player == player:
                if payoff > 0:
                    node.wins += 1
            node = node.parent
    
    # 返回访问次数最多的子节点对应的行动
    best_child = max(root.children, key=lambda x: x.visits, default=None)
    return best_child.action if best_child else None


# 简单的双人零和博弈示例
class SimpleGameState:
    def __init__(self, value=0, depth=0, max_depth=3):
        self.value = value
        self.depth = depth
        self.max_depth = max_depth
        self.history = []
    
    def copy(self):
        new_state = SimpleGameState(self.value, self.depth, self.max_depth)
        new_state.history = self.history.copy()
        return new_state

def get_actions(state: SimpleGameState) -> List[str]:
    return ["+1", "+2", "-1", "-2"]

def apply_action(state: SimpleGameState, action: str) -> SimpleGameState:
    new_state = state.copy()
    delta = int(action)
    new_state.value += delta
    new_state.depth += 1
    new_state.history.append(action)
    return new_state

def is_terminal(state: SimpleGameState) -> bool:
    return state.depth >= state.max_depth

def get_payoff(state: SimpleGameState) -> int:
    # Player1 希望 value 越大越好，Player2 相反
    return state.value

print("\n=== MCTS 示例：简单双人博弈 ===")
initial_state = SimpleGameState(value=0, max_depth=4)
best_action = mcts(
    initial_state, "Player1", num_iterations=500,
    get_actions=get_actions,
    apply_action=apply_action,
    is_terminal=is_terminal,
    get_payoff=get_payoff
)
print(f"MCTS 推荐的最佳行动：{best_action}")


=== MCTS 示例：简单双人博弈 ===
MCTS 推荐的最佳行动：+2


## 7. 总结

### 核心脉络

```
逆向归纳法（精确） 
    ↓
组合爆炸问题
    ↓  
MCTS（近似）
```

| 维度 | 逆向归纳法 | MCTS |
|------|-----------|------|
| **求解方式** | 遍历整棵树，精确计算 | 随机采样，统计估计 |
| **适用场景** | 小规模博弈树 | 大规模博弈树 |
| **理论基础** | 博弈论（SPE） | 强化学习 + 大数定律 |
| **计算复杂度** | $O(b^d)$ | $O(n)$，n 为模拟次数 |

### 三个层次的统一

1. **数据结构是骨架**：博弈树本质上就是一种树形数据结构，节点存储决策者信息，边存储行动，叶子节点存储支付。

2. **博弈论是灵魂**：逆向归纳法赋予了这棵树"理性"的内核——每个节点上的玩家都会选择最大化自己收益的行动。

3. **算法是桥梁**：DFS、MCTS 等算法将两者连接，使得我们可以用计算机求解复杂的动态博弈。

### 核心代码回顾

```python
# 逆向归纳法的核心逻辑（仅 15 行）
def backward_induction(node):
    if node.is_terminal():
        return node.payoff
    
    player_idx = get_player_index(node.player)
    best = None
    
    for child in node.children:
        payoff = backward_induction(child)
        if best is None or payoff[player_idx] > best[player_idx]:
            best = payoff
    
    return best
```

```python
# MCTS 的核心思想（UCT 公式）
UCT =  exploitation + exploration
     = (wins / visits) + c * sqrt(2 * ln(parent_visits) / visits)
```

### 最后的思考

每一个 if-else 逻辑判断，本质上都是博弈树上的一个分叉。当你下次写嵌套的 if-else 时，不妨想想：这可能是在求解某个动态博弈的子博弈精炼纳什均衡呢！